In [ ]:
import os, random, torch
from PIL import Image
from tqdm import tqdm   # <-- Jupyter için daha uyumlu tqdm
from torchvision import transforms
from medmnist import INFO
import medmnist
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import StableDiffusionImg2ImgPipeline

# =========================================================
# === CONFIG ==============================================
# =========================================================
MODEL_NAME = "Manojb/stable-diffusion-2-1-base"
EMBED_PATH = "./outputs/ti_embeddings/derma_embeds.pt"
LORA_PATH = "./outputs/lora_derma_finetune/exp_2025-11-05_02-27-44_lr0.0005_bs16_steps3000_rank32"
exp_name = os.path.basename(LORA_PATH.rstrip("/"))
OUTPUT_DIR = os.path.join("./outputs/ti_lora_generated_images", exp_name, "_t2i_generated")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STRENGTH = 0.65
GUIDANCE_SCALE = 2
STEPS = 40
N_PER_CLASS = 2000
BATCH_SIZE = 40
SAVE_INTERVAL = 500

torch.set_grad_enabled(False)

In [ ]:
def load_derma_embeddings(embed_path, tokenizer, text_encoder, device, dtype=torch.float16):
    """
    Load textual inversion embeddings into EXISTING tokenizer and text_encoder.
    Avoids reloading from Hub to prevent HTTP 404 errors.
    """
    embeds = torch.load(embed_path, map_location="cpu")
    added = 0
    for token, tensor in embeds.items():
        if token not in tokenizer.get_vocab():
            tokenizer.add_tokens(token)
            added += 1

    if added > 0:
        text_encoder.resize_token_embeddings(len(tokenizer))

    embed_layers = text_encoder.get_input_embeddings()
    embed_dtype = embed_layers.weight.dtype
    
    for token, tensor in embeds.items():
        token_id = tokenizer.convert_tokens_to_ids(token)
        embed_layers.weight.data[token_id] = tensor.to(device=device, dtype=embed_dtype)

    print(f"[INFO] Loaded {len(embeds)} textual inversion tokens (added {added}).")
    return tokenizer, text_encoder

In [ ]:
from diffusers import StableDiffusionPipeline 
import torch
import os

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
).to(DEVICE)

pipe.load_lora_weights(LORA_PATH)

pipe_dtype = next(pipe.unet.parameters()).dtype

load_derma_embeddings(
    embed_path=EMBED_PATH, 
    tokenizer=pipe.tokenizer, 
    text_encoder=pipe.text_encoder, 
    device=DEVICE, 
    dtype=pipe_dtype
)

pipe.enable_attention_slicing()
pipe.unet.to(memory_format=torch.channels_last)
pipe.vae.to(memory_format=torch.channels_last)
pipe.text_encoder.to(memory_format=torch.channels_last)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# =========================================================
# === LOAD DermMNIST ======================================
# =========================================================
info = INFO["dermamnist"]
DataClass = getattr(medmnist, info["python_class"])
base_ds = DataClass(split="train", download=True, size=224)
label_dict = {int(k): v.lower() for k, v in info["label"].items()}
classes = list(label_dict.values())

to_pil = transforms.Compose([
    transforms.Lambda(lambda x: x if isinstance(x, Image.Image) else transforms.functional.to_pil_image(x)),
    transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BICUBIC)
])

prompt_templates = [
    "a dermoscopic image of {} on the skin",
    "a dermoscopy photo showing {} on the skin",
    "a magnified dermoscopic image of {} lesion",
]

def random_prompt(class_name):
    return random.choice(prompt_templates).format(f"<{class_name}_lesion>")

In [ ]:
from tqdm import tqdm
import torch, os, random
from PIL import Image

print(f"[INFO] Generating {N_PER_CLASS} images per class × {len(classes)} classes = {N_PER_CLASS * len(classes)} total")
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()
pipe.vae.to(memory_format=torch.channels_last)
pipe.unet.to(memory_format=torch.channels_last)

total_images = N_PER_CLASS * len(classes)
progress_bar = tqdm(total=total_images, desc="Overall progress", ncols=100)


for cls_idx, cls_name in enumerate(classes):
    print(f"\n[CLASS] generating: {cls_name}")
    cls_out = os.path.join(OUTPUT_DIR, cls_name)
    os.makedirs(cls_out, exist_ok=True)

    rng = torch.Generator(device=DEVICE).manual_seed(0)
    generated_count = 0

    n_batches = (N_PER_CLASS + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_id in range(n_batches):

        current_batch_size = min(BATCH_SIZE, N_PER_CLASS - generated_count)

        prompts = [random_prompt(cls_name) for _ in range(current_batch_size)]

        with torch.no_grad(), torch.autocast("cuda", dtype=pipe.unet.dtype):

            results = pipe(
                prompt=prompts,
                guidance_scale=GUIDANCE_SCALE,
                num_inference_steps=STEPS,
                generator=rng,
                height=512, 
                width=512
            ).images

        for j, img in enumerate(results):
            img_id = batch_id * BATCH_SIZE + j

            save_path = os.path.join(cls_out, f"{cls_name}_{img_id:05d}.jpg")
            img.save(save_path)
            
            generated_count += 1
            progress_bar.update(1)

            if generated_count % SAVE_INTERVAL == 0:
                pass

    torch.cuda.empty_cache()
    print(f"[DONE] {cls_name}: {generated_count} images saved -> {cls_out}")

progress_bar.close()
print(f"\n All synthetic images generated successfully in: {OUTPUT_DIR}")

In [ ]:
from tqdm import tqdm
import torch, os, random
from PIL import Image


print(f"[INFO] Generating {N_PER_CLASS} images per class x {len(classes)} classes")

pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()
pipe.vae.to(memory_format=torch.channels_last)
pipe.unet.to(memory_format=torch.channels_last)

total_images = N_PER_CLASS * len(classes)
progress_bar = tqdm(total=total_images, desc="Overall progress", ncols=100)

for cls_idx, cls_name in enumerate(classes):
    print(f"\n[CLASS] Checking: {cls_name}")
    cls_out = os.path.join(OUTPUT_DIR, cls_name)
    os.makedirs(cls_out, exist_ok=True)

    existing_files = [f for f in os.listdir(cls_out) if f.endswith(".jpg")]
    num_existing = len(existing_files)

    progress_bar.update(num_existing)

    if num_existing >= N_PER_CLASS:
        print(f"   -> Skipping {cls_name}, already completed ({num_existing}/{N_PER_CLASS}).")
        continue
    
    print(f"   -> Resuming {cls_name} from {num_existing} to {N_PER_CLASS}...")
    
    generated_count = num_existing

    while generated_count < N_PER_CLASS:

        current_batch_size = min(BATCH_SIZE, N_PER_CLASS - generated_count)

        current_seed = cls_idx * 100000 + generated_count
        rng = torch.Generator(device=DEVICE).manual_seed(current_seed)

        prompts = [random_prompt(cls_name) for _ in range(current_batch_size)]

        with torch.no_grad(), torch.autocast("cuda", dtype=pipe.unet.dtype):
            results = pipe(
                prompt=prompts,
                guidance_scale=GUIDANCE_SCALE,
                num_inference_steps=STEPS,
                generator=rng,
                height=512, 
                width=512
            ).images

        for j, img in enumerate(results):
            img_id = generated_count 
            save_path = os.path.join(cls_out, f"{cls_name}_{img_id:05d}.jpg")
            img.save(save_path)
            
            generated_count += 1
            progress_bar.update(1)

        torch.cuda.empty_cache()

    print(f"[DONE] {cls_name}: Completed with {generated_count} images -> {cls_out}")

progress_bar.close()
print(f"\n All synthetic images generated successfully in: {OUTPUT_DIR}")

In [ ]:
import torch

def _count_named_params(module, name_filter_fn=None, requires_grad_only=False):
    total = 0
    for n, p in module.named_parameters():
        if requires_grad_only and (not p.requires_grad):
            continue
        if name_filter_fn is None or name_filter_fn(n, p):
            total += p.numel()
    return total

@torch.no_grad()
def report_sd_lora_ti_params(pipe, embed_path=None, verbose=True):
    """
    Prints:
      - Total Stable Diffusion params (UNet + VAE + Text Encoder)
      - LoRA params (name contains 'lora') across UNet/VAE/Text Encoder
      - Text encoder token embedding table params
      - Textual Inversion (TI) params loaded from `embed_path` (num_tokens * embedding_dim)
      - Percentages useful for papers
    """

    # ----------------------------
    # 1) Backbone totals
    # ----------------------------
    unet_total = sum(p.numel() for p in pipe.unet.parameters())
    vae_total  = sum(p.numel() for p in pipe.vae.parameters())
    te_total   = sum(p.numel() for p in pipe.text_encoder.parameters())
    sd_total   = unet_total + vae_total + te_total

    # ----------------------------
    # 2) LoRA totals (robust: name-based)
    # ----------------------------
    is_lora = lambda n, p: ("lora" in n.lower())
    unet_lora = _count_named_params(pipe.unet, is_lora, requires_grad_only=False)
    vae_lora  = _count_named_params(pipe.vae,  is_lora, requires_grad_only=False)
    te_lora   = _count_named_params(pipe.text_encoder, is_lora, requires_grad_only=False)
    lora_total = unet_lora + vae_lora + te_lora

    # ----------------------------
    # 3) Text token embedding table params
    # ----------------------------
    emb_layer = pipe.text_encoder.get_input_embeddings()
    vocab_size = emb_layer.weight.shape[0]
    emb_dim    = emb_layer.weight.shape[1]
    te_embedding_table_params = emb_layer.weight.numel()

    # ----------------------------
    # 4) TI params from embed_path
    # ----------------------------
    ti_tokens = 0
    ti_params = 0
    if embed_path is not None:
        embeds = torch.load(embed_path, map_location="cpu")
        # expected: dict(token_str -> tensor([emb_dim]) or (1, emb_dim))
        ti_tokens = len(embeds)
        ti_params = ti_tokens * emb_dim

    # ----------------------------
    # 5) Percentages for reporting
    # ----------------------------
    lora_pct_of_sd = (lora_total / sd_total * 100.0) if sd_total else 0.0
    ti_pct_of_sd   = (ti_params / sd_total * 100.0) if sd_total else 0.0
    peft_total     = lora_total + ti_params
    peft_pct_of_sd = (peft_total / sd_total * 100.0) if sd_total else 0.0

    # ----------------------------
    # 6) Print report
    # ----------------------------
    if verbose:
        print("=======================================================")
        print("Stable Diffusion Parameter Report (UNet + VAE + TextEnc)")
        print("=======================================================")
        print(f"UNet params:                 {unet_total:,}")
        print(f"VAE params:                  {vae_total:,}")
        print(f"Text Encoder params:         {te_total:,}")
        print("-------------------------------------------------------")
        print(f"TOTAL SD params:             {sd_total:,}")
        print("")
        print("=======================================================")
        print("LoRA Parameter Report (name contains 'lora')")
        print("=======================================================")
        print(f"UNet LoRA params:            {unet_lora:,}")
        print(f"VAE LoRA params:             {vae_lora:,}")
        print(f"Text Encoder LoRA params:    {te_lora:,}")
        print("-------------------------------------------------------")
        print(f"TOTAL LoRA params:           {lora_total:,}")
        print(f"LoRA as % of SD total:       {lora_pct_of_sd:.6f}%")
        print("")
        print("=======================================================")
        print("Text Embedding Table (CLIP token embeddings)")
        print("=======================================================")
        print(f"Vocab size:                  {vocab_size:,}")
        print(f"Embedding dim:               {emb_dim:,}")
        print(f"Embedding table params:      {te_embedding_table_params:,}")
        print("")
        print("=======================================================")
        print("Textual Inversion (TI) Report")
        print("=======================================================")
        if embed_path is None:
            print("TI embed_path:               None (not counted)")
        else:
            print(f"TI embed_path:               {embed_path}")
            print(f"TI tokens loaded:            {ti_tokens:,}")
            print(f"TI params (tokens*dim):      {ti_params:,}")
            print(f"TI as % of SD total:         {ti_pct_of_sd:.6f}%")
        print("")
        print("=======================================================")
        print("PEFT Summary (LoRA + TI)")
        print("=======================================================")
        print(f"TOTAL PEFT params:           {peft_total:,}")
        print(f"PEFT as % of SD total:       {peft_pct_of_sd:.6f}%")
        print("=======================================================")

    return {
        "unet_total": unet_total,
        "vae_total": vae_total,
        "text_encoder_total": te_total,
        "sd_total": sd_total,
        "unet_lora": unet_lora,
        "vae_lora": vae_lora,
        "text_encoder_lora": te_lora,
        "lora_total": lora_total,
        "lora_pct_of_sd": lora_pct_of_sd,
        "vocab_size": vocab_size,
        "emb_dim": emb_dim,
        "text_embedding_table_params": te_embedding_table_params,
        "ti_tokens": ti_tokens,
        "ti_params": ti_params,
        "ti_pct_of_sd": ti_pct_of_sd,
        "peft_total": peft_total,
        "peft_pct_of_sd": peft_pct_of_sd,
    }

# ----------------------------
# USAGE
# ----------------------------
report = report_sd_lora_ti_params(pipe, embed_path=EMBED_PATH)
